# 2-bosqich. ML uchun tayyorgarlik: targetlar va featurelar tahlili

**Maqsad:** guruch sarfi (`rice_kg`) va tushum (`revenue_uzs`) ga qaysi omillar qanchalik ta'sir qilishini,
omillarning o'zaro bog'liqligini aniqlash va modelga kiradigan featurelar ro'yxatini asoslash.

**Metodologik qoidalar**
1. **Faqat train ma'lumoti (2023–2024, 731 kun).** 2025-yil (169 kun) 3-bosqichdagi yakuniy test uchun saqlanadi.
   Feature tanlash test ma'lumotini ko'rib qilinsa, model aniqligi sun'iy ravishda yaxshi ko'rinadi.
2. **Leakage yo'q.** `customers` va `salad_count` kun oxirida ma'lum bo'ladi, shuning uchun ular X ga kirmaydi.
   Bitta target ikkinchisi uchun ham feature bo'lmaydi.
3. **Bashorat ufqi: 1–7 kun.** Feature bashorat qilinayotgan kundan oldin ma'lum bo'lishi kerak.
   Kalendar va tadbirlar oldindan ma'lum. Ob-havo uchun prognoz ishlatiladi deb faraz qilinadi
   (datasetda haqiqiy ob-havo bor, shuning uchun real hayotdagi aniqlik biroz pastroq bo'ladi).

**Tarkib**
- A. Targetlar tahlili: taqsimot, log transformatsiya, dekompozitsiya, statsionarlik, avtokorrelyatsiya
- B. X → Y: korrelyatsiya, mutual information, regressiyada sof ta'sir
- C. X ↔ X: korrelyatsiya, kovariatsiya, Cramér's V, multikollinearlik (VIF)
- D. Lag featurelar va interaksiyalar kerakmi
- E. Yakuniy feature ro'yxati

In [1]:
import sys
sys.path.insert(0, "..")

import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import statsmodels.api as sm
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, pacf, adfuller

from src import style
from src.data import load_data, TARGETS, BINARY, WEATHER_NUM, WEEKDAYS_UZ, MONTHS_UZ
from src.features import build_features, add_lag_features, train_test_split_time, FEATURES

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", module="statsmodels")
style.setup_notebook()
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = load_data()
train, test = train_test_split_time(df)
print(f"Train: {train.date.min():%Y-%m-%d} — {train.date.max():%Y-%m-%d} ({len(train)} kun)")
print(f"Test:  {test.date.min():%Y-%m-%d} — {test.date.max():%Y-%m-%d} ({len(test)} kun) — bu bosqichda ishlatilmaydi")

NAMES = {"temp_c": "Harorat", "humidity": "Namlik", "wind": "Shamol", "rain": "Yomg'ir", "snow": "Qor",
         "is_holiday": "Bayram", "ramadan": "Ramazon", "eid": "Hayit", "salary_week": "Maosh haftasi",
         "promotion": "Aksiya", "local_event": "Mahalliy tadbir", "football_match": "Futbol o'yini",
         "weekday": "Hafta kuni", "month": "Oy", "is_weekend": "Dam olish kuni", "season": "Fasl",
         **{f"wd_{i}": WEEKDAYS_UZ[i] for i in range(1, 7)}, **{f"m_{i}": MONTHS_UZ[i - 1] for i in range(2, 13)}}
TNAMES = {"rice_kg": "Guruch", "revenue_uzs": "Tushum"}

Train: 2023-01-01 — 2024-12-31 (731 kun)
Test:  2025-01-01 — 2025-06-18 (169 kun) — bu bosqichda ishlatilmaydi


## A. Targetlar tahlili

### A1. Taqsimot va log transformatsiya

In [2]:
rows = []
for t in TARGETS:
    for name, s in [("asl", train[t]), ("log", np.log(train[t]))]:
        rows.append({"target": TNAMES[t], "shkala": name, "skewness": s.skew(), "kurtosis": s.kurt(),
                     "Shapiro–Wilk p": stats.shapiro(s).pvalue})
pd.DataFrame(rows)

,target,shkala,skewness,kurtosis,Shapiro–Wilk p
0,Guruch,asl,0.850,1.137,0.000
1,Guruch,log,-0.157,0.105,0.161
2,Tushum,asl,0.800,1.191,0.000
3,Tushum,log,-0.224,0.133,0.028


**Xulosa.** Logarifmdan keyin qiyshiqlik deyarli yo'qoladi (skewness 0.8 → −0.1/−0.2).
Ikkinchi sabab ham bor: omillar ta'siri **multiplikativ** ko'rinadi (masalan, aksiya talabni ~30% ga oshiradi, "+15 kg" ga emas).
Log shkalada bunday ta'sirlar qo'shiluvchi bo'ladi va regressiya koeffitsientlarini **foiz** sifatida o'qish mumkin.
Shuning uchun chiziqli modellar `log(y)` da quriladi.

### A2. Dekompozitsiya (STL): trend, haftalik mavsumiylik va qoldiq

In [3]:
s = train.set_index("date")["rice_kg"]
stl = STL(s, period=7, robust=True).fit()
parts = [("Asl qator", s), ("Trend (yillik mavsumiylik shu yerda)", stl.trend),
         ("Haftalik mavsumiylik", stl.seasonal), ("Qoldiq", stl.resid)]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=[p[0] for p in parts])
for i, (_, v) in enumerate(parts, start=1):
    fig.add_scatter(x=v.index, y=v.values, mode="lines", line=dict(width=1, color=style.BLUE if i < 4 else style.GREY),
                    showlegend=False, row=i, col=1)
fig.update_layout(title="Guruch sarfining STL dekompozitsiyasi (davr = 7 kun, train)", height=750)
style.save(fig, "02_stl")

In [4]:
var_total = s.var()
pd.Series({
    "Trend (yillik)": stl.trend.var() / var_total,
    "Haftalik mavsumiylik": stl.seasonal.var() / var_total,
    "Qoldiq": stl.resid.var() / var_total,
}, name="dispersiya ulushi").to_frame()

,dispersiya ulushi
Trend (yillik),0.264
Haftalik mavsumiylik,0.333
Qoldiq,0.330


**Xulosa.** Trend komponenti silliq yillik to'lqin: qishda past, yozda yuqori, ikki yil bir xil shaklda.
Uzoq muddatli o'sish yoki pasayish yo'q. Haftalik komponent barqaror.
Qoldiqdagi alohida cho'qqilar bayram, aksiya va tadbir kunlari; ular B bo'limida tushuntiriladi.

### A3. Statsionarlik va avtokorrelyatsiya

In [5]:
adf = adfuller(train["rice_kg"], autolag="AIC")
pd.Series({"ADF statistikasi": adf[0], "p-qiymat": adf[1], "5% kritik qiymat": adf[4]["5%"]}, name="ADF testi").to_frame()

,ADF testi
ADF statistikasi,-2.758
p-qiymat,0.064
5% kritik qiymat,-2.866


In [6]:
def acf_figure(series, title, name, nlags=35):
    n = len(series)
    ci = 1.96 / np.sqrt(n)
    a, p = acf(series, nlags=nlags)[1:], pacf(series, nlags=nlags)[1:]
    lags = np.arange(1, nlags + 1)
    fig = make_subplots(rows=1, cols=2, subplot_titles=["ACF", "PACF"])
    for j, vals in enumerate([a, p], start=1):
        colors = [style.BLUE if l % 7 == 0 else style.GREY for l in lags]
        fig.add_bar(x=lags, y=vals, marker_color=colors, showlegend=False, row=1, col=j)
        for sgn in (1, -1):
            fig.add_hline(y=sgn * ci, line=dict(color=style.RED, dash="dash", width=1), row=1, col=j)
    fig.update_xaxes(title_text="Lag (kun)")
    fig.update_yaxes(range=[-0.4, 1], row=1, col=1)
    fig.update_yaxes(range=[-0.4, 1], row=1, col=2)
    fig.update_layout(title=title, height=360)
    return style.save(fig, name)

acf_figure(train["rice_kg"], "Guruch sarfining avtokorrelyatsiyasi (ko'k — 7, 14, 21… lag; qizil — 95% chegara)", "02_acf_raw")

**Xulosa.**
- ADF testi p ≈ 0.06: 5% darajada statsionarlik tasdiqlanmadi, lekin chegaraga yaqin.
  Buning sababi uzoq muddatli trend emas, balki kuchli **yillik mavsumiylik**: 2 yillik qatorda bu to'lqin
  "o'zgaruvchan o'rtacha" kabi ko'rinadi. STL dagi trend ham ikki yilda bir xil shaklni takrorlaydi.
  Yillik mavsumiylik oy va harorat featurelari bilan modellanadi, shuning uchun differensiallash shart emas.
  D1 da model qoldiqlarining statsionar ekanini tekshiramiz.
- ACF 7, 14, 21… laglarda kuchli: haftalik mavsumiylik. Lekin bu "kechagi kun ertangi kunga ta'sir qiladi" degani emas.
  Bu avtokorrelyatsiya hafta kuni va mavsum ta'siridan kelib chiqishi mumkin.
  D bo'limida tekshiramiz: kalendar va ob-havo hisobga olingandan keyin avtokorrelyatsiya qoladimi.

## B. X → Y: omillarning targetlarga ta'siri

### B1. Bir o'lchamli bog'liqlik: korrelyatsiya va mutual information
- **Pearson r:** chiziqli bog'liqlik. Binary X uchun u **point-biserial** korrelyatsiyaga teng.
- **Spearman ρ:** monoton (darajali) bog'liqlik, outlierlarga chidamli.
- **Mutual information (MI):** har qanday, jumladan chiziqli bo'lmagan bog'liqlik. Kategoriyali `weekday` va `month` uchun ham ishlaydi.

In [7]:
cand = WEATHER_NUM + BINARY + ["weekday", "month"]
discrete = [c not in WEATHER_NUM for c in cand]
Xc = train[cand].astype(float)
rows = []
for t in TARGETS:
    mi = mutual_info_regression(Xc, train[t], discrete_features=discrete, random_state=0)
    for c, m in zip(cand, mi):
        rows.append({"feature": c, "target": t, "Pearson r": train[c].astype(float).corr(train[t]),
                     "Spearman ρ": stats.spearmanr(train[c], train[t]).statistic, "MI": m})
uni = pd.DataFrame(rows)
uni_wide = uni.pivot(index="feature", columns="target", values=["Pearson r", "Spearman ρ", "MI"])
uni_wide = uni_wide.reindex(uni_wide[("MI", "rice_kg")].sort_values(ascending=False).index)
uni_wide

Pearson r          Spearman ρ                  MI        
target         revenue_uzs rice_kg revenue_uzs rice_kg revenue_uzs rice_kg
feature                                                                   
month                0.136   0.134       0.157   0.162       0.205   0.184
weekday              0.462   0.474       0.479   0.494       0.165   0.163
temp_c               0.459   0.454       0.484   0.488       0.163   0.158
snow                -0.346  -0.346      -0.367  -0.373       0.091   0.099
humidity            -0.404  -0.395      -0.420  -0.424       0.090   0.091
ramadan             -0.275  -0.277      -0.303  -0.310       0.066   0.051
salary_week          0.327   0.260       0.319   0.248       0.081   0.044
promotion            0.268   0.279       0.202   0.203       0.019   0.031
local_event          0.157   0.151       0.147   0.140       0.004   0.011
eid                  0.124   0.139       0.104   0.111       0.004   0.004
football_match       0.151   0.132       0.139   0.124       0.004   0.000
is_holiday           0.062   0.063       0.056   0.050       0.000   0.000
rain                -0.164  -0.167      -0.166  -0.174       0.000   0.000
wind                 0.051   0.051       0.054   0.050       0.000   0.000

In [8]:
u = uni[uni.target == "rice_kg"].set_index("feature")
u = u.reindex(u["MI"].sort_values().index)
labels = [NAMES[c] for c in u.index]
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=["Pearson r (chiziqli; weekday, month — tartib raqami sifatida)", "Mutual information (har qanday bog'liqlik)"])
fig.add_bar(y=labels, x=u["Pearson r"], orientation="h", showlegend=False,
            marker_color=np.where(u["Pearson r"] >= 0, style.GREEN, style.RED), row=1, col=1)
fig.add_bar(y=labels, x=u["MI"], orientation="h", showlegend=False, marker_color=style.BLUE, row=1, col=2)
fig.update_layout(title="Guruch sarfi bilan bir o'lchamli bog'liqlik (train)", height=520)
style.save(fig, "02_univariate")

**Kuzatuvlar.**
- Eng kuchli bog'liqlik **oy**, **hafta kuni**, **harorat**, **namlik** va **qor** bilan.
- Namlik bilan korrelyatsiya kuchli (r ≈ −0.4). Lekin bu sof ta'sirmi yoki namlik haroratning "soyasi"mi?
  Buni B2 dagi ko'p omilli regressiya ko'rsatadi.
- Aksiya, tadbir va bayram kam uchragani uchun MI ularni past baholaydi, garchi ularning bir kunlik ta'siri katta bo'lsa ham.
  Bir o'lchamli ko'rsatkichlar faqat dastlabki saralash uchun yaraydi.

### B2. Sof ta'sir: ko'p omilli regressiya (OLS, `log y`)

Model: `log(y) = β₀ + Σ βᵢ·Xᵢ + ε`, bunda barcha omillar bir vaqtda hisobga olinadi.
`exp(βᵢ) − 1` qiymati boshqa omillar o'zgarmaganda Xᵢ ning y ga **foizli ta'siri** bo'ladi.
Standart xatolar heteroskedastiklikka chidamli (HC3), ishonch oralig'i 95%.

Bu yerda barcha nomzod featurelar ishlatiladi: `humidity` va `wind` ham kiradi, faqat
`is_weekend` va `season` kirmaydi, chunki ular `weekday` va `month` ning to'liq nusxasi (C bo'limida ko'rsatiladi).

In [9]:
def candidate_X(d):
    X = build_features(d, features="all")
    return sm.add_constant(X.astype(float))

def fit_ols(d, target):
    return sm.OLS(np.log(d[target]), candidate_X(d)).fit(cov_type="HC3")

ols = {t: fit_ols(train, t) for t in TARGETS}

def effect_table(m):
    ci = m.conf_int()
    return pd.DataFrame({"ta'sir, %": (np.exp(m.params) - 1) * 100,
                         "95% CI quyi": (np.exp(ci[0]) - 1) * 100,
                         "95% CI yuqori": (np.exp(ci[1]) - 1) * 100,
                         "p-qiymat": m.pvalues}).drop("const")

eff = {t: effect_table(m) for t, m in ols.items()}
summary = pd.DataFrame({TNAMES[t]: [m.rsquared, m.rsquared_adj, m.nobs] for t, m in ols.items()},
                       index=["R²", "adj. R²", "n"])
summary

,Guruch,Tushum
R²,0.913,0.925
adj. R²,0.909,0.922
n,731.000,731.000


In [10]:
order = ["temp_c", "humidity", "wind", "rain", "snow", "is_holiday", "ramadan", "eid", "salary_week",
         "promotion", "local_event", "football_match"] + [f"wd_{i}" for i in range(1, 7)] + [f"m_{i}" for i in range(2, 13)]
e = pd.concat({TNAMES[t]: eff[t].loc[order] for t in TARGETS}, axis=1)
e.round(3)

Guruch                                       Tushum  \
               ta'sir, % 95% CI quyi 95% CI yuqori p-qiymat ta'sir, %   
temp_c             0.196      -0.029         0.421    0.088     0.207   
humidity           0.042      -0.044         0.128    0.339     0.004   
wind              -0.101      -0.373         0.171    0.466    -0.090   
rain             -15.189     -16.728       -13.621    0.000   -15.373   
snow             -32.422     -34.567       -30.207    0.000   -32.049   
is_holiday        27.246      18.464        36.680    0.000    28.904   
ramadan          -36.227     -38.423       -33.952    0.000   -36.432   
eid               52.719      43.594        62.425    0.000    46.162   
salary_week       19.667      17.790        21.575    0.000    25.334   
promotion         34.364      30.208        38.653    0.000    33.204   
local_event       20.474      15.240        25.945    0.000    21.945   
football_match    14.287      10.655        18.039    0.000    17.686   
wd_1               8.178       5.393        11.037    0.000     6.343   
wd_2              13.047      10.038        16.140    0.000    12.148   
wd_3              19.291      16.067        22.605    0.000    18.582   
wd_4              46.589      42.668        50.618    0.000    44.493   
wd_5              68.540      64.368        72.817    0.000    65.763   
wd_6              39.054      35.326        42.885    0.000    37.188   
m_2                4.623       0.500         8.916    0.028     4.769   
m_3               33.331      27.059        39.911    0.000    34.481   
m_4               33.335      25.774        41.349    0.000    33.758   
m_5               28.254      19.925        37.160    0.000    28.343   
m_6               38.029      27.270        49.698    0.000    34.865   
m_7               38.932      27.286        51.643    0.000    37.733   
m_8               35.781      25.396        47.027    0.000    35.909   
m_9               19.324      11.357        27.861    0.000    20.234   
m_10              23.505      17.068        30.296    0.000    25.349   
m_11              27.297      21.803        33.039    0.000    26.100   
m_12               2.862      -1.136         7.021    0.163     2.384   

                                                   
               95% CI quyi 95% CI yuqori p-qiymat  
temp_c               0.001         0.413    0.049  
humidity            -0.078         0.086    0.921  
wind                -0.336         0.157    0.476  
rain               -16.918       -13.800    0.000  
snow               -34.190       -29.838    0.000  
is_holiday          21.032        37.289    0.000  
ramadan            -38.535       -34.257    0.000  
eid                 35.966        57.123    0.000  
salary_week         23.444        27.253    0.000  
promotion           29.692        36.811    0.000  
local_event         17.186        26.897    0.000  
football_match      14.283        21.191    0.000  
wd_1                 3.663         9.092    0.000  
wd_2                 9.326        15.043    0.000  
wd_3                15.567        21.675    0.000  
wd_4                40.696        48.393    0.000  
wd_5                61.600        70.034    0.000  
wd_6                33.832        40.628    0.000  
m_2                  0.877         8.811    0.016  
m_3                 28.540        40.697    0.000  
m_4                 26.692        41.218    0.000  
m_5                 20.312        36.910    0.000  
m_6                 24.904        45.621    0.000  
m_7                 27.032        49.335    0.000  
m_8                 26.112        46.468    0.000  
m_9                 12.557        28.434    0.000  
m_10                19.063        31.966    0.000  
m_11                21.139        31.265    0.000  
m_12                -1.130         6.023    0.186

In [11]:
# Coefficient plot: harorat, namlik, shamol 10 birlik o'zgarish uchun ko'rsatiladi (1 birlikda juda kichik)
show = ["rain", "snow", "is_holiday", "ramadan", "eid", "salary_week", "promotion", "local_event", "football_match",
        "temp_c", "humidity", "wind"] + [f"wd_{i}" for i in range(1, 7)]

def per_unit(m, c, k):
    b, (lo, hi) = m.params[c] * k, m.conf_int().loc[c] * k
    return (np.exp(b) - 1) * 100, (np.exp(lo) - 1) * 100, (np.exp(hi) - 1) * 100

label = {**NAMES, "temp_c": "Harorat (+10°C)", "humidity": "Namlik (+10%)", "wind": "Shamol (+10 m/s)",
         **{f"wd_{i}": f"{WEEKDAYS_UZ[i]} (Dush ga nisbatan)" for i in range(1, 7)}}
fig = go.Figure()
for k_t, (t, color, off) in enumerate([("rice_kg", style.BLUE, -0.18), ("revenue_uzs", style.ORANGE, 0.18)]):
    vals = [per_unit(ols[t], c, 10 if c in WEATHER_NUM else 1) for c in show]
    y = np.arange(len(show)) + off
    fig.add_scatter(x=[v[0] for v in vals], y=y, mode="markers", marker=dict(color=color, size=8), name=TNAMES[t],
                    error_x=dict(type="data", symmetric=False, array=[v[2] - v[0] for v in vals],
                                 arrayminus=[v[0] - v[1] for v in vals], color=color, thickness=1.5, width=0))
fig.add_vline(x=0, line=dict(color="#444", width=1))
fig.update_yaxes(tickvals=np.arange(len(show)), ticktext=[label[c] for c in show], autorange="reversed")
fig.update_layout(title="Omillarning sof ta'siri, % (OLS, log y, 95% CI; boshqa omillar o'zgarmas)",
                  xaxis_title="y ning o'zgarishi, %", height=620, legend=dict(orientation="h", y=1.05, x=1, xanchor="right"))
style.save(fig, "02_coef_plot")

### B3. Xom farq va sof ta'sirni solishtirish

Xom farq — belgi bor va yo'q kunlardagi o'rtacha guruch sarfining farqi (1-bosqichdagi kabi).
Sof ta'sir esa boshqa omillar nazorat qilingandagi farq. Ular orasidagi tafovut **confounding**
(aralashib ketgan ta'sir) qanchalik kuchli ekanini ko'rsatadi.

In [12]:
rows = []
for c in BINARY:
    a, b = train.loc[train[c], "rice_kg"], train.loc[~train[c], "rice_kg"]
    rows.append({"belgi": NAMES[c], "n": len(a), "xom farq, %": (a.mean() / b.mean() - 1) * 100,
                 "Mann–Whitney p": stats.mannwhitneyu(a, b).pvalue,
                 "sof ta'sir, %": eff["rice_kg"].loc[c, "ta'sir, %"],
                 "sof: 95% CI": f"[{eff['rice_kg'].loc[c, '95% CI quyi']:.0f}; {eff['rice_kg'].loc[c, '95% CI yuqori']:.0f}]",
                 "sof p": eff["rice_kg"].loc[c, "p-qiymat"]})
raw_vs = pd.DataFrame(rows).sort_values("sof ta'sir, %")
raw_vs

,belgi,n,"xom farq, %",Mann–Whitney p,"sof ta'sir, %",sof: 95% CI,sof p
3,Ramazon,60,-32.198,0.000,-36.227,[-38; -34],0.000
1,Qor,50,-43.533,0.000,-32.422,[-35; -30],0.000
0,Yomg'ir,144,-13.381,0.000,-15.189,[-17; -14],0.000
8,Futbol o'yini,35,20.431,0.001,14.287,[11; 18],0.000
5,Maosh haftasi,240,19.253,0.000,19.667,[18; 22],0.000
7,Mahalliy tadbir,36,23.185,0.000,20.474,[15; 26],0.000
2,Bayram,14,15.064,0.179,27.246,[18; 37],0.000
6,Aksiya,52,36.502,0.000,34.364,[30; 39],0.000
4,Hayit,4,61.929,0.003,52.719,[44; 62],0.000


In [13]:
fig = go.Figure()
for _, r in raw_vs.iterrows():
    fig.add_scatter(x=[r["xom farq, %"], r["sof ta'sir, %"]], y=[r["belgi"]] * 2, mode="lines",
                    line=dict(color="#bbb", width=2), showlegend=False, hoverinfo="skip")
fig.add_scatter(x=raw_vs["xom farq, %"], y=raw_vs["belgi"], mode="markers", name="Xom farq",
                marker=dict(color=style.GREY, size=10, symbol="circle-open", line=dict(width=2)))
fig.add_scatter(x=raw_vs["sof ta'sir, %"], y=raw_vs["belgi"], mode="markers", name="Sof ta'sir (OLS)",
                marker=dict(color=style.BLUE, size=10))
fig.add_vline(x=0, line=dict(color="#444", width=1))
fig.update_layout(title="Guruch sarfi: xom farq va sof ta'sir, %", xaxis_title="%", height=420,
                  legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
style.save(fig, "02_raw_vs_adjusted")

In [14]:
num = pd.DataFrame({
    "Pearson r (xom)": [train[c].corr(train["rice_kg"]) for c in WEATHER_NUM],
    "sof ta'sir, % (+10 birlik)": [per_unit(ols["rice_kg"], c, 10)[0] for c in WEATHER_NUM],
    "sof p": [ols["rice_kg"].pvalues[c] for c in WEATHER_NUM],
}, index=[NAMES[c] for c in WEATHER_NUM])
num

,Pearson r (xom),"sof ta'sir, % (+10 birlik)",sof p
Harorat,0.454,1.973,0.088
Namlik,-0.395,0.419,0.339
Shamol,0.051,-1.009,0.466


**Xulosalar (B bo'limi).**
- Model guruch sarfi o'zgarishining **~91%**, tushum o'zgarishining **~92–93%** ini tushuntiradi (train, log shkalada).
- Ikkala target uchun omillar ta'siri deyarli bir xil. Bu kutilgan natija: ikkalasi ham mijozlar soniga bog'liq.
  Bitta istisno bor: **maosh haftasida** tushum guruchdan ko'proq oshadi, ya'ni odamlar har bir tashrifda ko'proq sarflaydi.
- Eng kuchli ijobiy omillar: **Hayit**, **Shanba/Juma**, **aksiya**, **bayram**, **mahalliy tadbir**, **maosh haftasi**.
  Eng kuchli salbiy omillar: **Ramazon** (~−35%), **qor** (~−33%), **yomg'ir** (~−15%).
- **Namlik** xom korrelyatsiyada kuchli (r ≈ −0.4) ko'rinadi, lekin harorat va oy nazorat qilinsa, uning ta'siri nolga yaqin va
  statistik ahamiyatsiz. Uning xom korrelyatsiyasi **haroratning soyasi** edi. **Shamol** ham ahamiyatsiz.
- **Harorat** oy hisobga olingandan keyin kuchsiz (~+2% har 10°C ga) va faqat chegaraviy ahamiyatli
  (guruch: p ≈ 0.09, tushum: p ≈ 0.05). Haroratning asosiy ta'sirini oy (mavsum) allaqachon o'z ichiga oladi.
  C4 da ko'rsatiladiki, harorat oy bilan kuchli kollinear, bu esa koeffitsientning standart xatosini oshiradi.
- **Confounding misollari** (B3): qorning xom ta'siri −44%, sof ta'siri esa −32%, chunki qorli kunlar qishga, ya'ni
  talab tabiiy ravishda past davrga to'g'ri keladi. **Bayram** aksincha: xom farq ahamiyatsiz (+15%, p = 0.18),
  sof ta'sir esa +27% va ahamiyatli. Bayramlarning bir qismi qishki past mavsumga tushadi va bu ta'sirni "yashiradi".
- **Hayit** va **bayram** uchun ishonch oralig'i keng: namunasi kichik (n = 4 va 14).

## C. X ↔ X: featurelarning o'zaro munosabati

### C1. Korrelyatsiya matritsasi

In [15]:
xx_cols = WEATHER_NUM + BINARY + ["weekday", "is_weekend", "month"]
C = train[xx_cols].astype(float).corr()
lab = [NAMES[c] for c in xx_cols]
fig = go.Figure(go.Heatmap(z=C.values, x=lab, y=lab, zmin=-1, zmax=1, colorscale=style.DIV_SCALE, reversescale=True,
                           text=np.round(C.values, 2), texttemplate="%{text:.2f}", textfont=dict(size=9),
                           colorbar=dict(title="r")))
fig.update_yaxes(autorange="reversed")
fig.update_layout(title="Featurelar orasidagi Pearson korrelyatsiyasi (binary juftliklar uchun = phi koeffitsienti)",
                  height=620, width=760, margin=dict(l=120, b=120))
style.save(fig, "02_corr_X")

In [16]:
pairs = (C.where(np.triu(np.ones(C.shape, dtype=bool), k=1)).stack().rename("r").reset_index()
          .assign(abs_r=lambda d: d.r.abs()).sort_values("abs_r", ascending=False).head(8))
pairs.assign(level_0=pairs.level_0.map(NAMES), level_1=pairs.level_1.map(NAMES)).drop(columns="abs_r") \
     .rename(columns={"level_0": "feature 1", "level_1": "feature 2"}).reset_index(drop=True)

,feature 1,feature 2,r
0,Hafta kuni,Dam olish kuni,0.791
1,Harorat,Namlik,-0.786
2,Harorat,Qor,-0.371
3,Namlik,Qor,0.270
4,Ramazon,Oy,-0.262
5,Harorat,Oy,0.189
6,Namlik,Oy,-0.154
7,Qor,Oy,-0.123


### C2. Kovariatsiya matritsasi

Kovariatsiya `cov(X, Y) = E[(X − μx)(Y − μy)]` birliklarga bog'liq. Masalan, harorat–namlik kovariatsiyasi °C·% birligida o'lchanadi,
shuning uchun uning kattaligini bevosita solishtirib bo'lmaydi.
Standartlashtirilgan (z-score) o'zgaruvchilarning kovariatsiyasi korrelyatsiyaga teng: `corr(X, Y) = cov(X, Y) / (σx·σy)`.

In [17]:
num_cols = WEATHER_NUM + ["rice_kg"]
cov_raw = train[num_cols].cov()
z = (train[num_cols] - train[num_cols].mean()) / train[num_cols].std()
cov_std = z.cov()
display(cov_raw.rename(index=NAMES, columns=NAMES).style.format("{:,.2f}").set_caption("Kovariatsiya (asl birliklarda)"))
display(cov_std.rename(index=NAMES, columns=NAMES).style.format("{:,.3f}").set_caption("Standartlashtirilgan kovariatsiya = korrelyatsiya"))

,Harorat,Namlik,Shamol,rice_kg
Harorat,171.29,-135.27,1.42,95.18
Namlik,-135.27,172.92,-1.09,-83.32
Shamol,1.42,-1.09,7.91,2.28
rice_kg,95.18,-83.32,2.28,257.02


,Harorat,Namlik,Shamol,rice_kg
Harorat,1.000,-0.786,0.039,0.454
Namlik,-0.786,1.000,-0.030,-0.395
Shamol,0.039,-0.030,1.000,0.051
rice_kg,0.454,-0.395,0.051,1.000


### C3. Kategoriyali ustunlar: Cramér's V

Kategoriyali juftliklar uchun Cramér's V ishlatiladi: 0 — bog'liqlik yo'q, 1 — biri ikkinchisini to'liq belgilaydi.

In [18]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n, k = ct.values.sum(), min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k))

cat = ["weekday", "is_weekend", "month", "season"]
V = pd.DataFrame([[cramers_v(train[a], train[b]) for b in cat] for a in cat], index=cat, columns=cat)
V.rename(index=NAMES, columns=NAMES)

,Hafta kuni,Dam olish kuni,Oy,Fasl
Hafta kuni,1.000,1.000,0.028,0.013
Dam olish kuni,1.000,1.000,0.031,0.005
Oy,0.028,0.031,1.000,1.000
Fasl,0.013,0.005,1.000,1.000


### C4. Multikollinearlik: VIF va condition number

`VIF_i = 1 / (1 − R²_i)`, bunda R²_i — Xᵢ ni qolgan featurelar orqali tushuntirish darajasi.
Odatiy chegaralar: VIF > 5 — o'rtacha, VIF > 10 — kuchli multikollinearlik.
Uch xil to'plam solishtiriladi:
1. **Hammasi:** `is_weekend` va `season` ham kiritilgan.
2. **Nomzodlar:** `is_weekend` va `season` olib tashlangan (B2 dagi model).
3. **Yakuniy:** `humidity` va `wind` ham olib tashlangan.

In [19]:
def vif_table(X):
    Xc = sm.add_constant(X.astype(float))
    with np.errstate(divide="ignore", invalid="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        v = [variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])]
    return pd.Series(v, index=X.columns)

X_all = build_features(train, features="all").assign(
    is_weekend=train.is_weekend.astype(int)).join(pd.get_dummies(train.season, prefix="s", drop_first=True, dtype=int))
X_cand = build_features(train, features="all")
X_final = build_features(train)

vifs = {"1. Hammasi": vif_table(X_all), "2. Nomzodlar": vif_table(X_cand), "3. Yakuniy": vif_table(X_final)}
cond = {k: np.linalg.cond(sm.add_constant(X.astype(float)).values) for k, X in
        zip(vifs, [X_all, X_cand, X_final])}
vif_df = pd.DataFrame(vifs)
vif_df.loc[["temp_c", "humidity", "wind", "is_weekend", "wd_5", "wd_6", "m_7", "s_summer"]].replace(np.inf, np.nan) \
      .rename(index=NAMES).style.format("{:,.1f}", na_rep="∞ (mukammal kollinear)")

,1. Hammasi,2. Nomzodlar,3. Yakuniy
Harorat,16.2,16.2,15.2
Namlik,2.7,2.7,∞ (mukammal kollinear)
Shamol,1.0,1.0,∞ (mukammal kollinear)
Dam olish kuni,"1,000,799,917,193,443.5",∞ (mukammal kollinear),∞ (mukammal kollinear)
Shan,"1,000,799,917,193,443.5",1.7,1.7
Yak,"1,000,799,917,193,443.5",1.7,1.7
Iyul,"1,000,799,917,193,443.5",10.5,10.4
s_summer,"1,000,799,917,193,443.5",∞ (mukammal kollinear),∞ (mukammal kollinear)


In [20]:
pd.DataFrame({"VIF > 10 bo'lgan featurelar": [int((v > 10).sum()) for v in vifs.values()],
              "Maksimal VIF": [v.replace(np.inf, np.nan).max() if np.isfinite(v).all() else np.inf for v in vifs.values()],
              "Condition number": list(cond.values())}, index=list(vifs))

,VIF > 10 bo'lgan featurelar,Maksimal VIF,Condition number
1. Hammasi,16,"1,000,799,917,193,443.500","905,383,692,490,010,880.000"
2. Nomzodlar,2,16.247,"1,795.261"
3. Yakuniy,2,15.206,520.835


In [21]:
v3 = vifs["3. Yakuniy"].sort_values()
fig = go.Figure(go.Bar(x=v3.values, y=[NAMES[c] for c in v3.index], orientation="h",
                       marker_color=np.where(v3 > 10, style.RED, np.where(v3 > 5, style.ORANGE, style.BLUE))))
for x, txt in [(5, "5"), (10, "10")]:
    fig.add_vline(x=x, line=dict(color=style.GREY, dash="dash"), annotation_text=txt, annotation_position="top")
fig.update_layout(title="Yakuniy featurelar to'plamining VIF qiymatlari", xaxis_title="VIF", height=620)
style.save(fig, "02_vif")

In [22]:
# Haroratning yuqori VIF i qayerdan: harorat oylar bilan qanchalik tushuntiriladi?
r2_temp_month = sm.OLS(train.temp_c, sm.add_constant(X_final.filter(like="m_").astype(float))).fit().rsquared
print(f"Harorat dispersiyasining {r2_temp_month:.1%} i faqat oy bilan tushuntiriladi.")

Harorat dispersiyasining 93.0% i faqat oy bilan tushuntiriladi.


**Xulosalar (C bo'limi).**
- **Mukammal multikollinearlik:** `is_weekend` to'liq `weekday` dan, `season` esa to'liq `month` dan kelib chiqadi (Cramér's V = 1, VIF = ∞).
  Bunday holatda OLS koeffitsientlarini aniqlab bo'lmaydi, shuning uchun bu ikki ustun olib tashlanadi.
- **Harorat ↔ oy (va namlik):** kuchli, lekin mukammal bo'lmagan multikollinearlik. Harorat dispersiyasining ~93% i
  oy bilan tushuntiriladi, shuning uchun harorat va yozgi oy dummy'larining VIF qiymati 10 dan yuqori. Harorat bilan namlik
  orasida ham r ≈ −0.79.
  - Bashorat uchun bu muammo emas: ikkalasi ham modelda qoladi, chunki oy haroratdan tashqari qo'shimcha ma'lumot beradi
    (Ramazon siljishi, ta'til davrlari va hokazo).
  - Talqin uchun esa harorat koeffitsientini "oy ichidagi harorat ta'siri" deb o'qish kerak.
  - Ridge regressiya va daraxtga asoslangan modellar bunday kollinearlikka chidamli.
- **Binary belgilar deyarli mustaqil:** ular orasida |r| < 0.1. Ya'ni aksiya, tadbir, bayram va boshqalarning ta'sirlari bir-biriga aralashmagan.

## D. Lag featurelar va interaksiyalar kerakmi?

### D1. Qoldiqlarda avtokorrelyatsiya qoldimi?
A3 da xom qatorda kuchli haftalik avtokorrelyatsiya bor edi. Agar u faqat kalendar ta'siridan kelib chiqqan bo'lsa,
OLS qoldiqlarida avtokorrelyatsiya bo'lmasligi kerak.

In [22]:
resid = ols["rice_kg"].resid
acf_figure(resid, "OLS qoldiqlarining avtokorrelyatsiyasi (guruch, train)", "02_acf_resid")

In [23]:
lb = pd.concat({TNAMES[t]: acorr_ljungbox(ols[t].resid, lags=[1, 7, 14]) for t in TARGETS})
lb.columns = ["Ljung–Box Q", "p-qiymat"]
display(lb)
pd.DataFrame({TNAMES[t]: {"ADF statistikasi": adfuller(ols[t].resid)[0], "ADF p-qiymat": adfuller(ols[t].resid)[1]}
              for t in TARGETS}).T.rename_axis("OLS qoldiqlari")

Ljung–Box Q  p-qiymat
Guruch 1         1.192     0.275
       7         4.809     0.683
       14       12.095     0.599
Tushum 1         2.833     0.092
       7        10.893     0.143
       14       19.713     0.139

,ADF statistikasi,ADF p-qiymat
OLS qoldiqlari,,
Guruch,-28.080,0.000
Tushum,-12.320,0.000


### D2. Lag featurelarni qo'shish modelni yaxshilaydimi?
Ufq 1–7 kun bo'lgani uchun faqat **kamida 7 kun oldingi** qiymatlar ishlatilishi mumkin:
`lag7`, `lag14` va `7 kunlik o'rtacha (lag7 dan boshlab)`.

In [24]:
rows = []
for t in TARGETS:
    L = add_lag_features(df, t, min_lag=7).loc[train.index]
    ok = L.notna().all(axis=1)
    y = np.log(train.loc[ok, t])
    X0 = candidate_X(train)[ok]
    m0 = sm.OLS(y, X0).fit()
    m1 = sm.OLS(y, X0.join(np.log(L[ok]))).fit()
    f_test = m1.compare_f_test(m0)
    rows.append({"target": TNAMES[t], "R² (lagsiz)": m0.rsquared, "R² (lag bilan)": m1.rsquared,
                 "adj. R² (lagsiz)": m0.rsquared_adj, "adj. R² (lag bilan)": m1.rsquared_adj,
                 "F-test p (laglar birgalikda)": f_test[1]})
pd.DataFrame(rows)

,target,R² (lagsiz),R² (lag bilan),adj. R² (lagsiz),adj. R² (lag bilan),F-test p (laglar birgalikda)
0,Guruch,0.912,0.913,0.908,0.909,0.169
1,Tushum,0.924,0.924,0.921,0.921,0.498


### D3. Interaksiyalar
Ikki mantiqiy gipoteza tekshiriladi:
- aksiya dam olish kunida kuchliroq ta'sir qiladimi (`aksiya × dam olish`);
- yomg'irning ta'siri haroratga bog'liqmi (`yomg'ir × harorat`).

In [25]:
rows = []
for t in TARGETS:
    X0 = candidate_X(train)
    Xi = X0.assign(promo_x_weekend=(train.promotion & train.is_weekend).astype(float),
                   rain_x_temp=train.rain * train.temp_c)
    m = sm.OLS(np.log(train[t]), Xi).fit(cov_type="HC3")
    for c, nm in [("promo_x_weekend", "Aksiya × dam olish"), ("rain_x_temp", "Yomg'ir × harorat")]:
        rows.append({"target": TNAMES[t], "interaksiya": nm, "koeffitsient": m.params[c], "p-qiymat": m.pvalues[c]})
pd.DataFrame(rows)

,target,interaksiya,koeffitsient,p-qiymat
0,Guruch,Aksiya × dam olish,-0.006,0.877
1,Guruch,Yomg'ir × harorat,-0.000,0.722
2,Tushum,Aksiya × dam olish,0.002,0.952
3,Tushum,Yomg'ir × harorat,0.000,0.935


**Xulosalar (D bo'limi).**
- Kalendar, ob-havo va tadbirlar hisobga olingandan keyin **qoldiqlarda avtokorrelyatsiya yo'q** (Ljung–Box p > 0.05),
  qoldiqlar esa statsionar (ADF p < 0.01).
  Xom qatordagi haftalik avtokorrelyatsiya to'liq hafta kuni va mavsum ta'siridan kelib chiqqan.
- **Lag featurelar** R² ni deyarli o'zgartirmaydi va statistik ahamiyatsiz. Ular modelga kiritilmaydi.
  Muhim amaliy natija: model o'tgan kunlar ma'lumotiga bog'liq emas, shuning uchun
  **ertangi kun ham, 7 kun keyingi kun ham bir xil aniqlikda** bashorat qilinadi (ob-havo prognozi ma'lum bo'lsa).
- **Interaksiyalar** ahamiyatsiz: omillar ta'siri log shkalada qo'shiluvchi.
  Bu log-chiziqli modelning to'g'ri tanlanganini tasdiqlaydi.

## E. Yakuniy feature ro'yxati

In [26]:
decision = pd.DataFrame([
    ("temp_c", "✅ kiradi", "Oy ichidagi harorat o'zgarishini ushlaydi; ahamiyati chegaraviy (tushum p≈0.05, guruch p≈0.13), 3-bosqichda CV bilan tekshiriladi"),
    ("rain, snow", "✅ kiradi", "Kuchli salbiy ta'sir (−15%, −33%)"),
    ("is_holiday, eid, ramadan", "✅ kiradi", "Kuchli ta'sir; bayram va Hayit uchun namuna kichik"),
    ("salary_week, promotion, local_event, football_match", "✅ kiradi", "Ahamiyatli ijobiy ta'sir"),
    ("weekday (6 dummy)", "✅ kiradi", "Eng kuchli omillardan biri"),
    ("month (11 dummy)", "✅ kiradi", "Yillik mavsumiylik; haroratdan tashqari ma'lumot beradi"),
    ("humidity", "❌ olib tashlandi", "Sof ta'siri ahamiyatsiz; xom korrelyatsiyasi haroratning soyasi"),
    ("wind", "❌ olib tashlandi", "Bog'liqlik yo'q (bir o'lchamli ham, sof ham)"),
    ("is_weekend", "❌ olib tashlandi", "weekday bilan mukammal kollinear"),
    ("season", "❌ olib tashlandi", "month bilan mukammal kollinear"),
    ("customers, salad_count", "❌ leakage", "Kun oxirida ma'lum bo'ladi"),
    ("lag featurelar", "❌ olib tashlandi", "Qoldiqlarda avtokorrelyatsiya yo'q; F-test ahamiyatsiz"),
], columns=["feature", "qaror", "asos"])
decision

,feature,qaror,asos
0,temp_c,✅ kiradi,Oy ichidagi harorat o'zgarishini ushlaydi; aha...
1,"rain, snow",✅ kiradi,"Kuchli salbiy ta'sir (−15%, −33%)"
2,"is_holiday, eid, ramadan",✅ kiradi,Kuchli ta'sir; bayram va Hayit uchun namuna ki...
3,"salary_week, promotion, local_event, football_...",✅ kiradi,Ahamiyatli ijobiy ta'sir
4,weekday (6 dummy),✅ kiradi,Eng kuchli omillardan biri
5,month (11 dummy),✅ kiradi,Yillik mavsumiylik; haroratdan tashqari ma'lum...
6,humidity,❌ olib tashlandi,Sof ta'siri ahamiyatsiz; xom korrelyatsiyasi h...
7,wind,❌ olib tashlandi,"Bog'liqlik yo'q (bir o'lchamli ham, sof ham)"
8,is_weekend,❌ olib tashlandi,weekday bilan mukammal kollinear
9,season,❌ olib tashlandi,month bilan mukammal kollinear


In [27]:
X_final = build_features(train)
print(f"Yakuniy featurelar soni: {X_final.shape[1]}")
print(list(X_final.columns))
assert list(X_final.columns) == FEATURES
for t in TARGETS:
    m = sm.OLS(np.log(train[t]), sm.add_constant(X_final.astype(float))).fit(cov_type="HC3")
    print(f"{TNAMES[t]}: yakuniy modelda harorat +10°C → {(np.exp(m.params['temp_c'] * 10) - 1) * 100:+.1f}%, p = {m.pvalues['temp_c']:.3f}")
m_final = {t: sm.OLS(np.log(train[t]), sm.add_constant(X_final.astype(float))).fit() for t in TARGETS}
pd.DataFrame({TNAMES[t]: {"R² (nomzodlar)": ols[t].rsquared, "R² (yakuniy)": m_final[t].rsquared,
                          "adj. R² (nomzodlar)": ols[t].rsquared_adj, "adj. R² (yakuniy)": m_final[t].rsquared_adj,
                          "AIC (nomzodlar)": ols[t].aic, "AIC (yakuniy)": m_final[t].aic} for t in TARGETS})

Yakuniy featurelar soni: 27
['temp_c', 'rain', 'snow', 'is_holiday', 'ramadan', 'eid', 'salary_week', 'promotion', 'local_event', 'football_match', 'wd_1', 'wd_2', 'wd_3', 'wd_4', 'wd_5', 'wd_6', 'm_2', 'm_3', 'm_4', 'm_5', 'm_6', 'm_7', 'm_8', 'm_9', 'm_10', 'm_11', 'm_12']
Guruch: yakuniy modelda harorat +10°C → +1.7%, p = 0.135
Tushum: yakuniy modelda harorat +10°C → +2.0%, p = 0.049


,Guruch,Tushum
R² (nomzodlar),0.913,0.925
R² (yakuniy),0.913,0.925
adj. R² (nomzodlar),0.909,0.922
adj. R² (yakuniy),0.909,0.922
AIC (nomzodlar),"-1,283.647","-1,378.696"
AIC (yakuniy),"-1,286.163","-1,382.140"


**Yakuniy xulosa (2-bosqich).**
1. Targetlar `log` shkalada modellanadi: taqsimot simmetrik bo'ladi, ta'sirlar foizda talqin qilinadi.
2. Guruch va tushumga eng kuchli ta'sir qiluvchi omillar: **hafta kuni**, **oy/mavsum**, **Ramazon**, **qor**, **aksiya**, **Hayit**, **bayram**.
3. Mukammal multikollinear (`is_weekend`, `season`) va ahamiyatsiz (`humidity`, `wind`) ustunlar olib tashlandi.
   Soddaroq model R² ni deyarli yo'qotmaydi va AIC bo'yicha yaxshiroq.
4. Lag featurelar kerak emas. Shuning uchun **1 kunlik va 7 kunlik ufq uchun bitta model** ishlatiladi.
5. Yakuniy to'plam: **27 feature** (`src/features.py → FEATURES`).

**3-bosqichda:** shu featurelar bilan baseline, Ridge, Random Forest va Gradient Boosting modellari quriladi va
2025-yil test ma'lumotida baholanadi.